# Training — AstroCLIP ViT-L/16 (Bloque A · Domain Shift Mínimo)

Notebook reutilizable que ejecuta únicamente llamadas a funciones definidas en `src/`.
Encoder **ViT-L/16** preentrenado sobre 76 M imágenes DESI en bandas **g, r, z**
mediante MAE + CLIP contrastivo con espectros ópticos DESI EDR (AstroCLIP).

> **Referencia:** Parker et al. 2024 — *AstroCLIP: A Cross-Modal Foundation Model for Galaxies* — MNRAS 531, 4990 — arXiv:2310.03024  
> **Pesos:** `hf_hub:PolymathicAI/astroclip_image` (HuggingFace Hub)

## 0) Dependencias específicas

AstroCLIP usa `timm` para cargar el ViT-L/16. No requiere `zoobot` ni `transformers`.

In [ ]:
# Instalar timm si no está disponible (ejecutar solo la primera vez)
# !pip install timm huggingface_hub

try:
    import timm
    print(f"timm {timm.__version__} — OK")
except ImportError:
    raise ImportError("Ejecuta: poetry add timm huggingface-hub")


## 1) Imports y configuración básica

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim

# Asegurar que la raíz del repo está en sys.path (notebook ubicado en `training_models/`)
sys.path.append(str(Path("..").resolve()))

from src import (
    CONFIG,
    CATALOG_PATH,
    FITS_DIR,
    build_dataloaders,
    calculate_class_weights,
    compute_classification_metrics,
    create_model_dir,
    evaluate_and_save,
    generate_gradcam_visualization,
    load_catalog,
    save_learning_curves,
    save_training_checkpoint,
    save_training_metadata,
    set_global_seed,
    split_catalog,
    train_loop,
    get_device,
)

# Builder específico para este experimento
from src.galaxy_model_builders import build_astroclip_vit

print("Imports OK")


## 2) Semilla y dispositivo

> **Advertencia VRAM:** AstroCLIP ViT-L tiene ~307 M parámetros en el encoder.
> Con `freeze_backbone=True` solo se cargan en memoria para forward pass.
> Se recomienda GPU con ≥ 16 GB VRAM; en GPUs menores reduce `CONFIG["batch_size"]` a 16 o 8.

In [ ]:
set_global_seed(CONFIG["seed"])
device = get_device()
print(f"Using device: {device}")

# Verificar VRAM disponible
if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f"VRAM disponible: {vram_gb:.1f} GB")
    if vram_gb < 16:
        print("[AVISO] < 16 GB VRAM — considera reducir batch_size a 16 u 8")


## 3) Carga del catálogo y división estratificada

In [ ]:
from pathlib import Path

catalog_path = Path(CATALOG_PATH)
report_path = Path("..") / "data" / "processed" / "dataset_validation_report.csv"

df = load_catalog(catalog_path, report_path=report_path)
train_df, val_df, test_df = split_catalog(df, seed=CONFIG["seed"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 4) Datasets y DataLoaders

> **Nota importante — tamaño de imagen:**  
> AstroCLIP fue preentrenado con imágenes de **518×518** píxeles.  
> Si `CONFIG["image_size"]` difiere de 518, es necesario interpolar los positional embeddings del ViT.
> El `build_astroclip_vit` de `galaxy_model_builders.py` ya maneja esto internamente via el argumento `img_size` de timm.
> 
> Si tu pipeline usa un tamaño distinto, considera añadir un `Resize(518)` en las transforms del DataLoader.

In [ ]:
# Verificar tamaño de imagen configurado
img_size = CONFIG.get("image_size", 518)
if img_size != 518:
    print(f"[AVISO] CONFIG image_size={img_size} — AstroCLIP fue preentrenado a 518x518.")
    print("         Considera añadir transforms.Resize(518) en build_dataloaders.")
else:
    print(f"image_size={img_size} — compatible con AstroCLIP ✓")

train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = build_dataloaders(
    train_df,
    val_df,
    test_df,
    FITS_DIR,
    CONFIG,
)

print("Datasets and DataLoaders ready")


## 5) Construcción del modelo (AstroCLIP ViT-L/16)

| Parámetro | Valor | Descripción |
|---|---|---|
| `freeze_backbone` | `True` | Solo entrena la cabeza (~132K params) |
| `head_hidden` | 128 | Igual que ResNet-18 training_models |
| `head_dropout` | 0.3 | Igual que ResNet-18 training_models |
| encoder dim | 1024-d | Dimensión del CLS token ViT-L |

In [ ]:
model_name = "astroclip_vit_l16"

model, trainable_params, frozen_params = build_astroclip_vit(
    num_classes=CONFIG["num_classes"],
    freeze_backbone=CONFIG.get("freeze_backbone", True),
    head_hidden=128,
    head_dropout=0.3,
    device=device,
)

print(f"Modelo      : AstroCLIP ViT-L/16")
print(f"Preentren.  : 76M imágenes DESI g,r,z (MAE + CLIP contrastivo con espectros)")
print(f"Encoder dim : 1024-d (CLS token)")
print(f"Trainable params: {trainable_params:,} | Frozen params: {frozen_params:,}")


## 6) Pérdida y optimizador

Idéntico al training_models. Con `freeze_backbone=True` el optimizer solo ve la cabeza lineal (~132K params),
por lo que el entrenamiento es rápido incluso con el encoder ViT-L completo en memoria.

In [ ]:
class_weights = calculate_class_weights(train_df, CONFIG["num_classes"], device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

print("Criterion and optimizer ready")


## 7) Entrenamiento y guardado de artefactos

In [ ]:
model_dir = create_model_dir(model_name)
checkpoint_path = model_dir / "checkpoint.pth"
history_path = model_dir / "history.json"

if checkpoint_path.exists() and history_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    history = checkpoint.get("history", {})
    print(f"Loaded existing checkpoint from {checkpoint_path}")
else:
    model, history = train_loop(
        model, train_loader, val_loader, criterion, optimizer, device,
        epochs=CONFIG["epochs"]
    )
    save_training_checkpoint(
        model,
        optimizer,
        history,
        CONFIG,
        class_weights,
        trainable_params,
        frozen_params,
        model_dir,
    )

training_meta = {
    "model_name": model_name,
    "architecture": "AstroCLIP ViT-L/16",
    "pretrained_on": "76M DESI Legacy Survey images (g, r, z) via MAE + contrastive CLIP",
    "domain_shift": "minimal",
    "encoder_dim": 1024,
    "epochs": CONFIG["epochs"],
    "config": CONFIG,
    "trainable_params": trainable_params,
    "frozen_params": frozen_params,
    "dataset_split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "catalog_path": str(catalog_path),
    "fits_dir": str(FITS_DIR),
    "checkpoint_file": checkpoint_path.name,
    "history_file": history_path.name,
    "reference": "Parker et al. 2024, MNRAS 531 4990, arXiv:2310.03024",
    "weights_hub": "hf_hub:PolymathicAI/astroclip_image",
}

save_training_metadata(model_dir, history, training_meta)
print(f"Model artifacts saved under: {model_dir}")


## 8) Evaluación final en Test Set

In [ ]:
eval_dir = model_dir / "evaluation"
sample_names = test_df["name"].tolist()
metrics = evaluate_and_save(
    model,
    test_loader,
    device,
    compute_classification_metrics,
    eval_dir,
    sample_names=sample_names,
)

print("Test metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")


## 9) Curvas de aprendizaje

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


In [ ]:
learning_curves_path = save_learning_curves(history, model_dir)
print(f"Learning curves saved: {learning_curves_path}")

img = mpimg.imread(learning_curves_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Learning curves — AstroCLIP ViT-L/16")
plt.show()


## 10) Grad-CAM de ejemplo

> **Nota ViT:** Los Vision Transformers **no tienen capas convolucionales**, por lo que
> Grad-CAM clásico no aplica directamente. Las alternativas son:
> - **Attention rollout**: visualiza la atención agregada de todos los heads del ViT.
> - **GradCAM++ sobre el último bloque de atención** (soportado por `pytorch-grad-cam` >= 1.4).
> 
> Si `generate_gradcam_visualization` de tu `src/` usa Grad-CAM estándar sobre capas conv,
> puede que no sea compatible con ViT. En ese caso la celda imprimirá un aviso y saltará la visualización.

In [ ]:
gradcam_dir = model_dir / "gradcam"
gradcam_dir.mkdir(parents=True, exist_ok=True)
gradcam_path = gradcam_dir / "gradcam_positive_label1.png"

try:
    saved_path = generate_gradcam_visualization(
        model, test_dataset, device, target_label=1, save_path=gradcam_path
    )
    print(f"Grad-CAM saved: {saved_path}")

    img = mpimg.imread(saved_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Grad-CAM for label 1 — AstroCLIP ViT-L/16")
    plt.show()

except Exception as e:
    print(f"[INFO] Grad-CAM no disponible para ViT: {e}")
    print("Para visualización en ViT considera 'attention rollout' o GradCAM++ con pytorch-grad-cam >= 1.4")


## 11) Notas

- Este notebook no define funciones nuevas: usa utilidades reutilizables de `src/` más `galaxy_model_builders.py`.
- **VRAM:** con `freeze_backbone=True` el encoder ViT-L (~307M params) está en memoria pero no acumula gradientes, lo que reduce el uso de VRAM frente a fine-tuning completo.
- **Fine-tuning completo:** si deseas descongelar el backbone, usa `freeze_backbone=False` y baja el learning rate a `1e-5` o `5e-6`. Requiere GPU ≥ 40 GB (A100/H100) para batch sizes razonables.
- **Bandas g, r, z:** AstroCLIP fue preentrenado exactamente con las mismas bandas de tu dataset, por lo que no es necesario ningún preprocesamiento especial de canales.
- Comparación directa con training_models: los artefactos se guardan en `model_dir` con el mismo esquema de directorios que `resnet18_training_models`.
